# M2 — bounded Dataset 2 audit

This notebook audits the prepared scatimdata Dataset 2 bundle before M3 decides the prediction cutoff, feature allowlist, or evaluation partitions. It is descriptive only: correlations and experiment effect sizes are not predictive performance or causal effects, audit trajectory summaries are not an M5 feature contract, and no records are removed. Signal amplitude units, several scalar units, integral/state timing, exact chronology, and the experiment 15 moisture discrepancy remain unresolved.

Source: Bogedale et al. (2023), *Online Prediction of Molded Part Quality in the Injection Molding Process Using High-Resolution Time Series*, DOI 10.3390/polym15040978; published release `sc4t1m/scatimdata` commit `7bd35941d75c97a3f276439377dc430ab47402be`, CC BY 4.0. See `docs/datasets/injection-molding-source-contract.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display

from mpi.datasets.injection_molding_persistence import load_bundle

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
BUNDLE_PATH = ROOT / "data/processed/injection_molding/dataset2"
bundle = load_bundle(BUNDLE_PATH)
plt.style.use("seaborn-v0_8-whitegrid")
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(14)

## Integrity snapshot and analysis frame

In [ ]:
weight = bundle.quality.filter(pl.col("characteristic") == "weight").select(
    "unit_id", pl.col("measured_value").alias("weight_g")
)
cycles = (
    bundle.units.select("unit_id", "cycle_counter", "source_row_index")
    .join(bundle.context, on="unit_id")
    .join(bundle.process_features, on=["unit_id", "operation_id"])
    .join(weight, on="unit_id")
    .sort("source_row_index")
)
experiment_counts = cycles.group_by("experiment_id").len().sort("experiment_id")
samples_per_cycle = bundle.signals.group_by("unit_id").len()["len"]
snapshot = pl.DataFrame(
    {
        "check": [
            "Labeled cycles",
            "Experiment 15",
            "Experiment 20",
            "Experiment 23",
            "Pressure samples/cycle",
            "Flow samples/cycle",
            "Signal-only excluded",
            "Weight missing",
        ],
        "value": [
            cycles.height,
            *[
                experiment_counts.filter(pl.col("experiment_id") == e)["len"][0]
                for e in (15, 20, 23)
            ],
            int(samples_per_cycle.min()),
            int(samples_per_cycle.min()),
            len(bundle.metadata.exclusions),
            weight["weight_g"].null_count(),
        ],
    }
)
display(snapshot)

## Weight target

The Tukey rule flags observations outside `[Q1 − 1.5×IQR, Q3 + 1.5×IQR]`, calculated overall and separately within each experiment. Flags are descriptive; flagged records remain in every analysis. Sample standard deviations (`ddof=1`) are reported.

In [ ]:
def descriptive(values: np.ndarray) -> dict[str, float | int]:
    values = values[np.isfinite(values)]
    q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    return {
        "n": len(values),
        "mean_g": values.mean(),
        "median_g": median,
        "std_g": values.std(ddof=1),
        "iqr_g": q3 - q1,
        "min_g": values.min(),
        "max_g": values.max(),
        "tukey_outliers": int(((values < low) | (values > high)).sum()),
    }


weight_rows = [{"scope": "overall", **descriptive(cycles["weight_g"].to_numpy())}]
for experiment in (15, 20, 23):
    values = cycles.filter(pl.col("experiment_id") == experiment)["weight_g"].to_numpy()
    weight_rows.append({"scope": f"experiment {experiment}", **descriptive(values)})
weight_summary = pl.DataFrame(weight_rows)
display(weight_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(cycles["weight_g"].to_numpy(), bins=25, color="#4472C4", edgecolor="white")
axes[0].set(xlabel="Part weight (g)", ylabel="Cycles", title="Weight distribution")
groups = [cycles.filter(pl.col("experiment_id") == e)["weight_g"].to_numpy() for e in (15, 20, 23)]
axes[1].boxplot(groups, tick_labels=["15", "20", "23"], showfliers=True)
axes[1].set(xlabel="Experiment ID", ylabel="Part weight (g)", title="Weight by experiment")
plt.tight_layout()
plt.show()

## Process-scalar inventory, associations, redundancy, and experiment shift

The 31 retained process scalars are audited, not approved as predictors. A constant has one observed value. A near-constant has more than one observed value but at least 95% of non-null rows equal its modal value. Pearson associations use pair-complete rows and are shown overall and within experiments. Pairwise standardized mean difference is `(mean_a − mean_b) / sqrt((sample_variance_a + sample_variance_b) / 2)` on non-null rows. A zero pooled denominator yields 0 for equal means and signed infinity otherwise. These effect sizes describe experiment dependence; they are not causal.

In [ ]:
feature_names = bundle.process_features.columns[2:]


def finite_pair(a: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mask = np.isfinite(a) & np.isfinite(b)
    return a[mask], b[mask]


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    a, b = finite_pair(a.astype(float), b.astype(float))
    return float(np.corrcoef(a, b)[0, 1]) if len(a) > 1 and a.std() > 0 and b.std() > 0 else np.nan


def smd(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    denominator = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    if denominator == 0:
        return 0.0 if a.mean() == b.mean() else float(np.sign(a.mean() - b.mean()) * np.inf)
    return float((a.mean() - b.mean()) / denominator)


pairs = [(15, 20), (15, 23), (20, 23)]
overview_rows, distribution_rows = [], []
for name in feature_names:
    series = cycles[name]
    observed = series.drop_nulls().to_numpy()
    counts = series.drop_nulls().value_counts().sort("count", descending=True)
    correlations = {}
    for experiment in (15, 20, 23):
        subset = cycles.filter(pl.col("experiment_id") == experiment)
        correlations[f"r_exp_{experiment}"] = pearson(
            subset[name].to_numpy(), subset["weight_g"].to_numpy()
        )
        vals = subset[name].drop_nulls().to_numpy()
        distribution_rows.append(
            {
                "field": name,
                "experiment_id": experiment,
                "n": len(vals),
                "mean": vals.mean() if len(vals) else np.nan,
                "std": vals.std(ddof=1) if len(vals) > 1 else np.nan,
                "q25": np.quantile(vals, 0.25) if len(vals) else np.nan,
                "median": np.median(vals) if len(vals) else np.nan,
                "q75": np.quantile(vals, 0.75) if len(vals) else np.nan,
                "min": vals.min() if len(vals) else np.nan,
                "max": vals.max() if len(vals) else np.nan,
            }
        )
    shifts = {
        f"{a}_vs_{b}": smd(
            cycles.filter(pl.col("experiment_id") == a)[name].to_numpy(),
            cycles.filter(pl.col("experiment_id") == b)[name].to_numpy(),
        )
        for a, b in pairs
    }
    finite_shifts = {key: value for key, value in shifts.items() if not np.isnan(value)}
    largest_pair = max(finite_shifts, key=lambda key: abs(finite_shifts[key]))
    overview_rows.append(
        {
            "field": name,
            "missing": series.null_count(),
            "unique": series.n_unique(),
            "mean": observed.mean(),
            "std": observed.std(ddof=1),
            "min": observed.min(),
            "max": observed.max(),
            "constant": len(counts) == 1,
            "near_constant": len(counts) > 1 and counts["count"][0] / len(observed) >= 0.95,
            "r_overall": pearson(series.to_numpy(), cycles["weight_g"].to_numpy()),
            **correlations,
            "max_abs_smd": abs(finite_shifts[largest_pair]),
            "largest_shift_pair": largest_pair,
        }
    )
scalar_overview = pl.DataFrame(overview_rows).sort("max_abs_smd", descending=True)
scalar_distributions = pl.DataFrame(distribution_rows)
display(
    scalar_overview.select(
        "field",
        "missing",
        "unique",
        "constant",
        "near_constant",
        "r_overall",
        "r_exp_15",
        "r_exp_20",
        "r_exp_23",
        "max_abs_smd",
        "largest_shift_pair",
    )
)
for experiment in (15, 20, 23):
    print(f"Full scalar distributions: experiment {experiment}")
    with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=120):
        display(scalar_distributions.filter(pl.col("experiment_id") == experiment))

redundant = []
for i, left in enumerate(feature_names):
    for right in feature_names[i + 1 :]:
        r = pearson(cycles[left].to_numpy(), cycles[right].to_numpy())
        if np.isfinite(r) and abs(r) >= 0.95:
            redundant.append({"left": left, "right": right, "r": r, "abs_r": abs(r)})
redundancy_table = (
    pl.DataFrame(redundant).sort("abs_r", descending=True)
    if redundant
    else pl.DataFrame({"note": ["No |r| >= 0.95 pairs"]})
)
display(redundancy_table)

selected_scalars = [
    "maximum_injection_pressure",
    "switchover_injection_pressure",
    "injection_time",
    "melt_cushion",
    "dosing_time",
    "mold_heating_circuit_1",
]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, name in zip(axes.flat, selected_scalars, strict=True):
    values = [cycles.filter(pl.col("experiment_id") == e)[name].to_numpy() for e in (15, 20, 23)]
    ax.boxplot(values, tick_labels=["15", "20", "23"], showfliers=False)
    ax.set(
        title=name.replace("_", " "),
        xlabel="Experiment ID",
        ylabel="Native value (unit unresolved)",
    )
plt.tight_layout()
plt.show()

## Context-separated weight variation

Experiment ID and the moisture, mold-temperature, and charge context remain outside the process table. The descriptive fraction below is between-experiment sum of squares divided by total weight sum of squares. It is not a causal effect or modeling score.

In [ ]:
all_weight = cycles["weight_g"].to_numpy()
grand_mean = all_weight.mean()
between_ss = sum(len(group) * (group.mean() - grand_mean) ** 2 for group in groups)
total_ss = ((all_weight - grand_mean) ** 2).sum()
weight_shift = [
    {
        "pair": f"{a} vs {b}",
        "weight_smd": smd(groups[(15, 20, 23).index(a)], groups[(15, 20, 23).index(b)]),
    }
    for a, b in pairs
]
display(
    pl.DataFrame(
        {
            "metric": ["between-experiment / total weight variation"],
            "value": [between_ss / total_ss],
        }
    )
)
display(pl.DataFrame(weight_shift))
display(
    cycles.select(
        "experiment_id", "mean_moisture_content", "mold_temperature", "source_charge_code"
    )
    .group_by("experiment_id")
    .agg(pl.all().null_count().name.suffix("_nulls"), pl.all().n_unique().name.suffix("_unique"))
    .sort("experiment_id")
)

## Pressure and flow trajectories

Four examples per channel and experiment are selected with seed 20260913. Curves use the native released elapsed-time grid (2,048 points, including three 0.004 s steps); no resampling occurs. Signal amplitude units are unknown. Envelopes are pointwise 10th–90th percentiles. AUC uses `numpy.trapezoid` with each cycle's actual elapsed times and therefore has units of native amplitude × seconds.

In [ ]:
n_cycles, n_samples = cycles.height, int(samples_per_cycle[0])
ordered_signals = bundle.signals.join(
    cycles.select("unit_id", "source_row_index"), on="unit_id", validate="m:1"
).sort("source_row_index", "sample_index")
signal_units = ordered_signals["unit_id"].to_numpy().reshape(n_cycles, n_samples)
assert np.array_equal(signal_units[:, 0], cycles["unit_id"].to_numpy())
time_matrix = ordered_signals["elapsed_time_seconds"].to_numpy().reshape(n_cycles, n_samples)
signal_matrices = {
    name: ordered_signals[name].to_numpy().reshape(n_cycles, n_samples)
    for name in ("injection_pressure", "injection_flow")
}
experiments = cycles["experiment_id"].to_numpy()
rng = np.random.default_rng(20260913)
fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for row, experiment in enumerate((15, 20, 23)):
    indices = np.flatnonzero(experiments == experiment)
    examples = rng.choice(indices, size=4, replace=False)
    for col, (name, matrix) in enumerate(signal_matrices.items()):
        ax = axes[row, col]
        for index in examples:
            ax.plot(time_matrix[index], matrix[index], color="0.7", linewidth=0.7, alpha=0.8)
        values, x = matrix[indices], time_matrix[indices[0]]
        q10, mean, q90 = (
            np.quantile(values, 0.10, axis=0),
            values.mean(axis=0),
            np.quantile(values, 0.90, axis=0),
        )
        ax.fill_between(x, q10, q90, color="#9DC3E6", alpha=0.45, label="10th-90th percentile")
        ax.plot(x, mean, color="#1F4E79", linewidth=1.4, label="mean")
        title = "Experiment {}: {}".format(experiment, name.replace("_", " "))
        ax.set(title=title, ylabel="Native amplitude (unit unresolved)")
        if row == 0 and col == 0:
            ax.legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("Elapsed time (s)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for ax, (name, matrix) in zip(axes, signal_matrices.items(), strict=True):
    for experiment in (15, 20, 23):
        indices = np.flatnonzero(experiments == experiment)
        ax.plot(time_matrix[indices[0]], matrix[indices].std(axis=0, ddof=1), label=str(experiment))
    title = "{}: cycle-to-cycle SD".format(name.replace("_", " "))
    ax.set(title=title, xlabel="Elapsed time (s)", ylabel="SD (native amplitude)")
    ax.legend(title="Experiment")
plt.tight_layout()
plt.show()

trajectory_rows = []
trajectory_features = {}
for name, matrix in signal_matrices.items():
    trajectory_features[f"{name}_mean"] = matrix.mean(axis=1)
    trajectory_features[f"{name}_max"] = matrix.max(axis=1)
    trajectory_features[f"{name}_auc"] = np.trapezoid(matrix, time_matrix, axis=1)
for experiment in (15, 20, 23):
    mask = experiments == experiment
    for metric, values in trajectory_features.items():
        trajectory_rows.append(
            {
                "experiment_id": experiment,
                "audit_summary": metric,
                "n": int(mask.sum()),
                "mean": values[mask].mean(),
                "std": values[mask].std(ddof=1),
                "median": np.median(values[mask]),
                "min": values[mask].min(),
                "max": values[mask].max(),
            }
        )
trajectory_summary = pl.DataFrame(trajectory_rows)
display(trajectory_summary)

## Observed intervention segments in source order

Segment boundaries below are changes in released context values, not inferred physical phase boundaries or calendar days. Experiments 15 and 20 show raw moisture; experiment 23 shows raw mold temperature. Experiment 15's raw values (0.050/0.100/0.150) disagree partly with the paper (0.066%/0.097%/0.150%); neither is repaired. Context nulls remain null.

In [ ]:
def runs(values: list[float | None]) -> list[tuple[int, int, float | None]]:
    result, start = [], 0
    for index in range(1, len(values) + 1):
        changed = index == len(values) or (
            values[index] != values[start] and not (values[index] is None and values[start] is None)
        )
        if changed:
            result.append((start, index - start, values[start]))
            start = index
    return result


segment_rows = []
for experiment, field in (
    (15, "mean_moisture_content"),
    (20, "mean_moisture_content"),
    (23, "mold_temperature"),
):
    subset = cycles.filter(pl.col("experiment_id") == experiment)
    for start, length, value in runs(subset[field].to_list()):
        segment_rows.append(
            {
                "experiment_id": experiment,
                "context_field": field,
                "start_within_experiment": start,
                "length": length,
                "raw_value": value,
            }
        )
segments = pl.DataFrame(segment_rows)
segment_summaries = []
for row in segments.iter_rows(named=True):
    subset = cycles.filter(pl.col("experiment_id") == row["experiment_id"]).slice(
        row["start_within_experiment"], row["length"]
    )
    segment_summaries.append(
        {
            **row,
            "weight_mean_g": subset["weight_g"].mean(),
            "maximum_injection_pressure_mean": subset["maximum_injection_pressure"].mean(),
            "injection_time_mean": subset["injection_time"].mean(),
        }
    )
segments = pl.DataFrame(segment_summaries)
display(segments)

fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex="row")
measurements = [
    ("weight_g", "Weight (g)"),
    ("maximum_injection_pressure", "Maximum injection pressure (unit unresolved)"),
    ("injection_time", "Injection time (unit unresolved)"),
]
for row, (experiment, field) in enumerate(
    ((15, "mean_moisture_content"), (20, "mean_moisture_content"), (23, "mold_temperature"))
):
    subset = cycles.filter(pl.col("experiment_id") == experiment)
    context_runs = runs(subset[field].to_list())
    x = np.arange(subset.height)
    for col, (measurement, label) in enumerate(measurements):
        ax = axes[row, col]
        ax.plot(x, subset[measurement].to_numpy(), linewidth=0.8)
        for start, length, value in context_runs:
            if start:
                ax.axvline(start, color="0.4", linestyle="--", linewidth=0.8)
            if col == 0:
                ax.text(
                    start + length / 2,
                    0.98,
                    f"{'moisture' if 'moisture' in field else 'mold temp'}={value:g}",
                    transform=ax.get_xaxis_transform(),
                    ha="center",
                    va="top",
                    fontsize=8,
                )
        ax.set(title=f"Experiment {experiment}", ylabel=label)
    axes[row, 1].set_xlabel("Labeled cycle position within experiment (source order)")
plt.tight_layout()
plt.show()

## Exhaustive source-field and availability inventory

Every one of the 40 source scalars remains individually traceable, followed by the required pressure/flow channels and the retained optional cavity/state groups. `Candidate process` means only that M3 may investigate the field; it is not an allowlist. Complete-cycle availability is a hypothesis, not an approved prediction cutoff.

In [ ]:
def field_assessment(section: str, source: str, canonical: str) -> tuple[str, str, str, str]:
    if source == "cycle_counter":
        return (
            "Machine-local cycle identity/order key",
            "Present in released scalar row; not a physical timestamp",
            "Identity and order proxy",
            "Outcome/identity exclusion",
        )
    if section == "quality":
        meaning = (
            "Part-weight outcome in grams"
            if source == "weight"
            else "Post-cycle geometry outcome; unit/scale unresolved"
        )
        return (
            meaning,
            "Measured after the molded part was removed",
            "Direct target or post-process outcome",
            "Outcome/identity exclusion",
        )
    if section == "context":
        meaning = {
            "Versuch": "Experiment/trial identifier",
            "mittlerer Feuchtegehalt": "Released mean-moisture context",
            "Twkz": "Released mold-temperature context",
            "Charge": "Released source charge code",
        }[source]
        return (
            meaning,
            "Recorded experimental context; deployment-time knowledge unresolved",
            "Strong domain/intervention proxy",
            "Context-only",
        )
    if source.startswith("integral_"):
        return (
            "Source-computed state-window integral; state semantics unresolved",
            "State/window timing relative to cutoff unresolved",
            "Possible post-cutoff information",
            "Unresolved/deferred",
        )
    return (
        canonical.replace("_", " "),
        "Retained cycle scalar; exact completion time unresolved until M3",
        "Possible post-cutoff value if cutoff is early",
        "Candidate process",
    )


inventory_rows = []
for lineage in bundle.metadata.scalar_lineage:
    meaning, availability, risk, disposition = field_assessment(
        lineage.canonical_section, lineage.source_name, lineage.canonical_name
    )
    inventory_rows.append(
        {
            "source_name": lineage.source_name,
            "canonical_name": lineage.canonical_name,
            "meaning": meaning,
            "availability_evidence": availability,
            "leakage_risk": risk,
            "provisional_m3_disposition": disposition,
        }
    )
for lineage in bundle.metadata.signal_lineage:
    inventory_rows.append(
        {
            "source_name": lineage.source_group,
            "canonical_name": lineage.canonical_field,
            "meaning": "Native-grid cycle trajectory; amplitude unit unresolved",
            "availability_evidence": "Full cycle present; cutoff unresolved until M3",
            "leakage_risk": "Post-cutoff samples if an earlier cutoff is selected",
            "provisional_m3_disposition": "Candidate process",
        }
    )
for group in bundle.metadata.retained_optional_source_groups:
    inventory_rows.append(
        {
            "source_name": group,
            "canonical_name": None,
            "meaning": "Retained optional cavity-pressure or state group; not decoded in M2",
            "availability_evidence": "Presence retained; timing/state semantics unresolved",
            "leakage_risk": "Unknown state windows and cutoff relationship",
            "provisional_m3_disposition": "Unresolved/deferred",
        }
    )
field_inventory = pl.DataFrame(inventory_rows)
assert field_inventory.head(40).height == 40
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=120):
    display(field_inventory)

## Audit checkpoint values

This compact final output supports transfer of aggregate evidence into the milestone document. M3 still owns all eligibility, cutoff, and split decisions.

In [ ]:
checkpoint = {
    "weight_overall": weight_rows[0],
    "between_experiment_weight_fraction": float(between_ss / total_ss),
    "constant_process_fields": scalar_overview.filter(pl.col("constant"))["field"].to_list(),
    "near_constant_process_fields": scalar_overview.filter(pl.col("near_constant"))[
        "field"
    ].to_list(),
    "top_process_shifts": scalar_overview.select("field", "max_abs_smd", "largest_shift_pair")
    .head(8)
    .to_dicts(),
    "strongest_absolute_overall_weight_associations": scalar_overview.with_columns(
        pl.col("r_overall").abs().alias("abs_r")
    )
    .sort("abs_r", descending=True)
    .select("field", "r_overall")
    .head(8)
    .to_dicts(),
    "trajectory_summary_by_experiment": trajectory_summary.to_dicts(),
}
checkpoint